# Klasifikasi Jenis Jalan (C2) pada Titik Genangan

Notebook ini menambahkan indikator **C2 (klasifikasi jenis jalan)** yang sebelumnya
belum ada di `severity/index.py`. Prosesnya: tarik seluruh ruas jalan Kota Surabaya
dari tag `highway` OpenStreetMap (trunk sampai service/living_street), lalu untuk
tiap titik genangan cari ruas jalan yang bersinggungan/terdekat dan petakan ke
skala ordinal 1-5.

**Skema pemetaan (sesuai Bab 2.2.3.1 / Tabel indikator SI proposal):**

| Tag `highway` | Nilai ordinal |
|---|---|
| trunk, primary | 5 |
| secondary | 4 |
| tertiary | 3 |
| residential | 2 |
| service, living_street | 1 |

Jika satu titik genangan berada dalam radius yang mencakup lebih dari satu ruas
jalan, dipakai nilai **tertinggi** (paling kritikal), sesuai catatan proposal.

**Output:** `road_classification_surabaya.csv` dengan kolom
`flood_id, road_class, highway_type_used, distance_to_road_m`, siap di-merge
ke `floods.csv` sebagai kolom baru dan dipakai di `severity/index.py`.

In [1]:
%pip install -q geopandas shapely requests pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import LineString, Point

# ===== KONFIGURASI =====
# Bounding box sama seperti main.py backend, supaya konsisten
BBOX = (-7.38, 112.58, -7.13, 112.87)  # south, west, north, east

# Kategori jalan yang ditarik: trunk sampai service/living_street
HIGHWAY_TYPES = [
    'trunk', 'primary', 'secondary', 'tertiary',
    'residential', 'service', 'living_street',
]

# Pemetaan ke skala ordinal (Persamaan indikator C2, Bab 2)
HIGHWAY_ORDINAL = {
    'trunk': 5, 'primary': 5,
    'secondary': 4,
    'tertiary': 3,
    'residential': 2,
    'service': 1, 'living_street': 1,
}

OVERPASS_ENDPOINTS = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://overpass.private.coffee/api/interpreter',
]
USER_AGENT = 'MDCVRP-IF-SI-TA/1.0 (student research; Surabaya flood pumping)'

CRS_WGS84 = 'EPSG:4326'
CRS_UTM49S = 'EPSG:32749'

# Radius pencarian ruas jalan di sekitar titik genangan (meter)
# Titik dalam radius ini dianggap "mencakup" ruas tsb -> ambil nilai tertinggi
SEARCH_RADIUS_M = 100

DATA_DIR = Path("../backend/app/data")
OUTPUT_DIR = Path("output_road_class")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup selesai.")

Setup selesai.


## 1. Query Overpass untuk Seluruh Kategori Jalan

Pola sama seperti notebook `3__Intermediate_Facilities_Surabaya.ipynb`
(fetch dengan retry dan fallback mirror), tapi kategori highway diperluas
sampai residential dan service/living_street.

In [3]:
def fetch_overpass(query, max_retries=3, retry_wait=30):
    headers = {'User-Agent': USER_AGENT, 'Accept': 'application/json'}
    last_response = None
    for endpoint in OVERPASS_ENDPOINTS:
        print(f'Trying endpoint: {endpoint}')
        for attempt in range(1, max_retries + 1):
            try:
                response = requests.post(endpoint, data={'data': query}, headers=headers, timeout=180)
                last_response = response
                if response.status_code == 200:
                    return response.json()
                print(f'  Attempt {attempt}: HTTP {response.status_code}')
            except requests.RequestException as e:
                print(f'  Attempt {attempt}: {type(e).__name__}: {e}')
            time.sleep(retry_wait)
        print('  All retries exhausted, switching endpoint')
    if last_response is not None:
        last_response.raise_for_status()
    raise RuntimeError('Semua Overpass endpoint gagal')


def build_highway_query(highway_types, bbox):
    south, west, north, east = bbox
    values_regex = '|'.join(highway_types)
    return f'''
    [out:json][timeout:180];
    (
      way["highway"~"^({values_regex})$"]({south},{west},{north},{east});
    );
    out geom tags;
    '''.strip()


highway_query = build_highway_query(HIGHWAY_TYPES, BBOX)
highway_raw = fetch_overpass(highway_query)
print(f"Jumlah ruas jalan: {len(highway_raw['elements'])}")

Trying endpoint: https://overpass-api.de/api/interpreter
Jumlah ruas jalan: 79662


## 2. Parse ke GeoDataFrame

In [4]:
def elements_to_gdf(elements):
    records = []
    for el in elements:
        if el.get('type') != 'way' or 'geometry' not in el:
            continue
        coords = [(pt['lon'], pt['lat']) for pt in el['geometry']]
        if len(coords) < 2:
            continue
        tags = el.get('tags', {})
        records.append({
            'osm_id': el['id'],
            'name': tags.get('name'),
            'highway_type': tags.get('highway'),
            'geometry': LineString(coords),
        })
    return gpd.GeoDataFrame(records, crs=CRS_WGS84)


highway_gdf = elements_to_gdf(highway_raw['elements'])
highway_gdf['ordinal'] = highway_gdf['highway_type'].map(HIGHWAY_ORDINAL)
highway_gdf = highway_gdf.dropna(subset=['ordinal']).reset_index(drop=True)

print(f"Ruas jalan valid (punya pemetaan ordinal): {len(highway_gdf)}")
print(highway_gdf['highway_type'].value_counts())

highway_utm = highway_gdf.to_crs(CRS_UTM49S)

Ruas jalan valid (punya pemetaan ordinal): 79662
highway_type
residential      41871
service          20821
living_street    12199
tertiary          1749
secondary         1237
primary            997
trunk              788
Name: count, dtype: int64


## 3. Muat Titik Genangan dan Cari Ruas Jalan dalam Radius

Untuk tiap titik genangan: cari semua ruas jalan dalam `SEARCH_RADIUS_M`,
ambil ordinal **tertinggi** di antaranya. Kalau tidak ada ruas dalam radius,
perluas ke ruas terdekat (berapapun jaraknya) sebagai fallback, supaya tidak
ada titik yang kosong nilainya.

In [7]:
floods_df = pd.read_csv(DATA_DIR)
floods_df = floods_df.rename(columns={"Latitude": "lat", "Longitude": "lon"})
floods_df["lat"] = pd.to_numeric(floods_df["lat"], errors="coerce")
floods_df["lon"] = pd.to_numeric(floods_df["lon"], errors="coerce")
floods_df = floods_df.dropna(subset=["lat", "lon"]).reset_index(drop=True)

if "id" not in floods_df.columns:
    floods_df.insert(0, "id", [f"F_{i:04d}" for i in range(len(floods_df))])

floods_gdf = gpd.GeoDataFrame(
    floods_df,
    geometry=gpd.points_from_xy(floods_df["lon"], floods_df["lat"]),
    crs=CRS_WGS84,
).to_crs(CRS_UTM49S)

print(f"Titik genangan: {len(floods_gdf)}")

Titik genangan: 34


In [10]:
def classify_point(point_geom, roads_gdf, radius_m):
    """Cari ordinal tertinggi di antara ruas dalam radius; fallback ke ruas terdekat."""
    buffer = point_geom.buffer(radius_m)
    within = roads_gdf[roads_gdf.intersects(buffer)]

    if len(within) > 0:
        best_idx = within['ordinal'].idxmax()
        best = within.loc[best_idx]
        dist = point_geom.distance(best.geometry)
        return best['ordinal'], best['highway_type'], dist, True  # True = dalam radius

    # Fallback: ruas terdekat, berapapun jaraknya
    distances = roads_gdf.geometry.distance(point_geom)
    nearest_idx = distances.idxmin()
    nearest = roads_gdf.loc[nearest_idx]
    return nearest['ordinal'], nearest['highway_type'], distances.loc[nearest_idx], False


results = []
for _, row in floods_gdf.iterrows():
    ordinal, hw_type, dist, in_radius = classify_point(row.geometry, highway_utm, SEARCH_RADIUS_M)
    results.append({
        'flood_id': row['id'],
        'road_class': int(ordinal),
        'highway_type_used': hw_type,
        'distance_to_road_m': round(float(dist), 2),
        'within_search_radius': in_radius,
    })

road_class_df = pd.DataFrame(results)
print(road_class_df['road_class'].value_counts().sort_index())
print()
n_fallback = (~road_class_df['within_search_radius']).sum()
print(f"Titik yang pakai fallback (di luar radius {SEARCH_RADIUS_M}m): {n_fallback} dari {len(road_class_df)}")
road_class_df.head(10)

road_class
1     3
2    12
3    10
4     6
5     3
Name: count, dtype: int64

Titik yang pakai fallback (di luar radius 100m): 0 dari 34


,flood_id,road_class,highway_type_used,distance_to_road_m,within_search_radius
0,F_0000,2,residential,98.73,True
1,F_0001,2,residential,69.75,True
2,F_0002,4,secondary,2.39,True
3,F_0003,5,trunk,93.32,True
4,F_0004,3,tertiary,35.27,True
5,F_0005,3,tertiary,51.74,True
6,F_0006,2,residential,74.17,True
7,F_0007,2,residential,0.00,True
8,F_0008,2,residential,39.91,True
9,F_0009,4,secondary,55.63,True


## 4. Simpan Output

In [11]:
road_class_df.to_csv(OUTPUT_DIR / "road_classification_surabaya.csv", index=False)
print(f"Tersimpan: {OUTPUT_DIR}/road_classification_surabaya.csv")

# Preview merge ke floods.csv (tidak menimpa file asli di sini, cukup preview)
merged_preview = floods_df.merge(
    road_class_df[['flood_id', 'road_class']],
    left_on='id', right_on='flood_id', how='left'
).drop(columns=['flood_id'])
merged_preview.head()

Tersimpan: output_road_class/road_classification_surabaya.csv


,id,no,tanggal,jalan,kecamatan,kelurahan,area,detail,depth_cm,lat,lon,terima_berita,tiba_tkp,mulai_penanganan,selesai,road_class
0,F_0000,377,2026-06-22,SWK Ketintang dan Jl. Lusi,Gayungan,Ketintang,Surabaya Selatan,Genangan : 5 cm >> Surut>>,5.0,-7.316784,112.729173,2026-06-22 05:14:00,2026-06-22 05:20:00,2026-06-22 05:21:00,2026-06-22 09:22:00,2
1,F_0001,378,2026-06-22,Wisma Tengger,Benowo,Kandangan,Surabaya Barat,Genangan : 10 cm = Surut Clear 0%,10.0,-7.262527,112.656512,2026-06-22 05:14:00,2026-06-22 05:20:00,2026-06-22 05:21:00,2026-06-22 08:14:00,2
2,F_0002,379,2026-06-22,SMPN 12,Wonokromo,Ngagel Rejo,Surabaya Selatan,Genangan : 20 cm,20.0,-7.297865,112.749232,2026-06-22 05:57:00,2026-06-22 06:02:00,2026-06-22 06:03:00,2026-06-22 10:05:00,4
3,F_0003,380,2026-06-22,JL. Bung Tomo Ngagel dan Jl. Ngagel Timur,Wonokromo,Ngagel,Surabaya Selatan,Genangan : 20 cm,20.0,-7.290469,112.745126,2026-06-22 06:31:00,2026-06-22 06:37:00,2026-06-22 06:38:00,2026-06-22 09:15:00,5
4,F_0004,381,2026-06-22,Gereja SMTB,Gubeng,Baratajaya,Surabaya Timur,Genangan : 5 cm,5.0,-7.288979,112.760230,2026-06-22 06:43:00,2026-06-22 06:49:00,2026-06-22 06:50:00,2026-06-22 09:30:00,3


## Catatan Penting

- Kolom `road_class` (nilai 1-5) ini yang jadi input C2 di `severity/index.py`.
  Saat ini `severity/index.py` cuma pakai `depth_cm` dan `dist_faskes_m`
  (2 kolom di `CRITERIA`); perlu ditambahkan `road_class` sebagai kolom ketiga,
  dan matriks pairwise AHP (`PAIRWISE`) perlu diperluas dari 2x2 jadi 3x3
  begitu bobot hasil wawancara sudah ada.
- `SEARCH_RADIUS_M = 100` meter adalah asumsi awal (jarak toleransi genangan
  dianggap "berada di" suatu ruas jalan). Sesuaikan kalau perlu lebih ketat/longgar,
  dan laporkan nilai yang dipakai di Bab 4.2.1 sebagai bagian dari desain.
- Titik yang jatuh ke fallback (`within_search_radius = False`) berarti tidak
  ada ruas jalan berkategori trunk-service dalam radius pencarian; ini bisa
  terjadi kalau ada gap data OSM. Layak dicek manual satu-satu sebelum dipakai
  di eksperimen final.